In [2]:
import numpy as np
import pandas as pd

In [3]:
from moddata import load_data

In [4]:
from tsfresh.feature_extraction import extract_features, MinimalFCParameters
from tsfresh.utilities.dataframe_functions import impute
from tsfresh.feature_extraction import EfficientFCParameters
from tsfresh import select_features

## 1. Load Data

In [5]:
data = load_data("sunspots")

In [6]:
data["day"] = data["day"].astype("datetime64[ns]")
data = data.set_index("day")
data["y"] = data["daily_sunspots_number"].shift(-1)

In [7]:
data = data["2011-01-01":"2015-12-31"]
data = data.reset_index(drop=False)

In [8]:
X = data[["day", "daily_sunspots_number"]]
y = pd.Series(data=data["y"].values, index=data.index)

In [15]:
X.head(3)

,day,daily_sunspots_number
0,2011-01-01,48.0
1,2011-01-02,48.0
2,2011-01-03,47.0


In [18]:
y.head()

0    48.0
1    47.0
2    48.0
3    36.0
4    28.0
dtype: float64

## 2. Manually Transform to Rolling Format

In [10]:
window_size = 60

In [11]:
chunks: list[pd.DataFrame] = []
for j in range(window_size-1, X.shape[0], 1):
    chunk = data.iloc[j-(window_size-1):j+1, :]
    chunk["window_id"] = j
    chunk["window_time"] = np.arange(window_size)
    chunks.append(chunk)

In [12]:
X_rolling = pd.concat(chunks, axis=0)

In [20]:
X_rolling.head(3)

,day,daily_sunspots_number,y,window_id,window_time
0,2011-01-01,48.0,48.0,59,0
1,2011-01-02,48.0,47.0,59,1
2,2011-01-03,47.0,48.0,59,2


In [21]:
X_rolling.tail(3)

,day,daily_sunspots_number,y,window_id,window_time
1823,2015-12-29,57.0,37.0,1825,57
1824,2015-12-30,37.0,22.0,1825,58
1825,2015-12-31,22.0,37.0,1825,59


In [ ]:
features = extract_features(
    rolling_sunspots.drop(columns=["day"]),
    column_id="window_id",
    column_sort="window_time",
    default_fc_parameters=EfficientFCParameters(),
    # n_jobs=1,
    disable_progressbar=False,
)